# Electricity load exploration

This notebook imports the installed project package. Dataset preparation, feature definitions, evaluation and training live in tested modules. Run `uv sync --locked` and preparation first; select this project's `.venv` Python kernel.

Task: a new one-hour-ahead forecast after every observed hour, using customer MT_158. The source's Portuguese wall-clock/DST convention is preserved; target values are hourly mean kW. [Source and attribution](https://doi.org/10.24432/C58C86).

In [ ]:
from IPython.display import Image, display
from electricity_demo.evaluate import plot_forecast  # Initializes a local plot cache.
import matplotlib.pyplot as plt
from electricity_demo.config import load_training_config, project_path
from electricity_demo.prepare import load_hourly
from electricity_demo.features import build_features
from electricity_demo.split import split_development_test, chronological_folds

config = load_training_config('configs/ridge.yaml')
data = load_hourly(project_path(config['dataset']), project_path(config['dataset_manifest']))
print(f'{len(data):,} validated hourly rows')
data.head()

In [ ]:
ax = data.iloc[:336].plot(x='timestamp', y='load_kw', figsize=(12, 3), title='Development period: first two weeks')
ax.set_ylabel('Average load (kW)')
plt.show()

In [ ]:
features = build_features(data, config['features'])
development, final_test = split_development_test(features, config['evaluation'])
folds = chronological_folds(development, config['evaluation'])
print('Initial history rows dropped:', len(data) - len(features))
for number, (train, validation) in enumerate(folds, 1):
    print(f'Fold {number}: train through {development.index[train[-1]]}; validate {development.index[validation[0]]} through {development.index[validation[-1]]}')
print('Reserved final test:', final_test.index[0], 'through', final_test.index[-1])
features.head()

Training uses the exact same entry point as the terminal. This run evaluates **selection CV only**; it does not score the final test period. All fitted preprocessing is learned inside the corresponding training fold.

In [ ]:
from electricity_demo.train import run_training
run = run_training('configs/ridge.yaml')
display(Image(filename=str(run / 'forecast.png')))
print('Run artifacts:', run)

Use the project workflow for DVC recovery, the alternative model and automated checks. After selecting the model using development CV, explicitly pass `--final-test` to the training CLI once for the final evaluation. Use development CV when comparing dataset versions.

The estimator is fixed while hourly observed history advances. Earlier observed test hours provide history for subsequent one-step forecasts. The model is not forecasting the entire test block from one origin.